# 7. Porovnanie modelov

Všetky modely sú hodnotené rovnako: na rovnakých 10 deleniach train/test podľa zdrojového
videa. Ako referenciu pridávame model, ktorý vždy predpovedá najčastejšiu triedu
v trénovacích dátach (úroveň náhody: balanced accuracy 0,5).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from har_utils import make_splits, metrics, summarize

df = pd.read_csv("index.csv")
y = df.label.values

# referencia: vždy najčastejšia trieda z trénovacích dát
baseline = pd.DataFrame([
    {"split": seed, **metrics(y[te], np.full(len(te), np.bincount(y[tr]).argmax()))}
    for seed, tr, te in make_splits(df)
])

results = {
    "Najčastejšia trieda": baseline,
    "CNN-SVM (ResNet18)": pd.read_csv("results/cnn_svm.csv"),
    "CNN-LSTM (ResNet18)": pd.read_csv("results/cnn_lstm.csv"),
    "R(2+1)D + SVM": pd.read_csv("results/video_cnn_svm.csv"),
}
if Path("results/video_finetune.csv").exists():      # notebook 6b (doladenie siete)
    results["R(2+1)D doladená"] = pd.read_csv("results/video_finetune.csv")

table = pd.DataFrame([summarize(r, name) for name, r in results.items()])
table.columns = ["accuracy", "balanced accuracy", "F1 (macro)"]
table

In [ ]:
names = list(results)
plt.figure(figsize=(8, 4))
plt.boxplot([results[n].bal_acc for n in names], tick_labels=names)
plt.axhline(0.5, ls="--", c="gray")
plt.ylabel("balanced accuracy (10 delení)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()